# Week 12 build · Run, compare, decide, write up

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-12.ipynb)

**~3 hours · CPU only · run top to bottom.** `SMOKE = True` is a five-minute rehearsal; set it to `False` for the real run (still CPU, about five minutes of compute).

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running. The notebook refuses empty predictions. The first predict cell asks for the pre-registered X and the sharpened ablation before any data is loaded (canvas steps 6–9: *run*, *predict before running*, *log the surprise*, *decide*).

**Data:** upload `week10_data.npz` and `week10_plan.md` from week 10 to the Files panel. If they are missing, Part A regenerates the data with week 10's code and seed and uses the default plan; the write-up records which happened.

| Part | Topic | Time |
|---|---|---|
| A | Data, plan, and the synthetic listener | 25 min |
| B | Trivial baselines: pixel distance and fixed-mapping distance | 20 min |
| C | Train the contrastive model once; read the curves | 35 min |
| D | Four rows × seeds, rank correlation, bootstrap interval, the one plot | 45 min |
| E | Optional: the codec alternative (4×4, 8 levels, straight-through) | 25 min |
| F | The one-page write-up and the decision | 30 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

---
## Part A · Data, plan, and the synthetic listener (25 min)

**Words used in this part.** *SMOKE*: a flag that shrinks data, steps and seeds so the notebook runs in five minutes on CPU. *week10_data.npz*: the file week 10 saved (images, sound vectors, haptic codes, labels). *synthetic listener*: a fixed, noisy, coarse-eared decision rule that "hears" each sonified shape and answers with a class; a stand-in for the human study, **not** evidence about people. *confusion table*: 20 rows (played) × 20 columns (answered); folded into one count per shape pair.

In [ ]:
#@title Config: SMOKE = True is a five-minute CPU rehearsal; False is the full run (still CPU)
SMOKE = True  #@param {type:"boolean"}
RUN_CODEC = True  #@param {type:"boolean"}
import os, json, math, time, torch, torch.nn as nn, torch.nn.functional as F
cfg = dict(n_per_class=12 if SMOKE else 60, seeds=[0, 1, 2] if SMOKE else [0, 1, 2, 3, 4],
           steps=300 if SMOKE else 3000, listener_repeats=20 if SMOKE else 50, boots=1000, seed=0)
DATA_FILE, PLAN_FILE = "week10_data.npz", "week10_plan.md"
torch.manual_seed(cfg["seed"]); rng = np.random.default_rng(cfg["seed"]); T_START = time.time()
print("SMOKE:", SMOKE, "| codec part:", RUN_CODEC, "| config:", cfg)

In [ ]:
#@title 🔮 Predict A0 (before any data): copy the pre-registered X and the sharpened ablation from week10_plan.md, then predict whether A will clear X against the pixel-loss encoder
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
X = 0.2  #@param {type:"number"}
ablation_text = "drop the sound side: contrastive image<->image with jitter, same encoder, same steps"  #@param {type:"string"}
assert X > 0 and ablation_text.strip(), "X and the ablation must be written before any run"
predict("A0", prediction, confidence)
print(f"pre-registered: A beats B (pixel-loss encoder) by >= {X} rank correlation | ablation: {ablation_text}")

In [ ]:
yy, xx = np.mgrid[0:16, 0:16].astype(float)                     # week 10's renderer, repeated so the data can be regenerated
def _m(dx, dy, r):
    return dict(ax=np.abs(dx), ay=np.abs(dy), d2=dx**2 + dy**2, r=r, dx=dx, dy=dy)
SHAPES = {
 "circle":     lambda m: m["d2"] <= m["r"]**2,
 "ring":       lambda m: (m["d2"] <= m["r"]**2) & (m["d2"] >= (0.6 * m["r"])**2),
 "square":     lambda m: np.maximum(m["ax"], m["ay"]) <= m["r"],
 "hollow sq":  lambda m: (np.maximum(m["ax"], m["ay"]) <= m["r"]) & (np.maximum(m["ax"], m["ay"]) > 0.6 * m["r"]),
 "diamond":    lambda m: m["ax"] + m["ay"] <= m["r"],
 "tri up":     lambda m: (m["ay"] <= m["r"]) & (m["ax"] <= (m["dy"] + m["r"]) * 0.6),
 "tri down":   lambda m: (m["ay"] <= m["r"]) & (m["ax"] <= (m["r"] - m["dy"]) * 0.6),
 "plus":       lambda m: ((m["ax"] <= 1.2) & (m["ay"] <= m["r"])) | ((m["ay"] <= 1.2) & (m["ax"] <= m["r"])),
 "X":          lambda m: (np.abs(m["ax"] - m["ay"]) <= 1.2) & (np.maximum(m["ax"], m["ay"]) <= m["r"]),
 "h bar":      lambda m: (m["ay"] <= 1.2) & (m["ax"] <= m["r"]),
}

In [ ]:
SHAPES.update({
 "v bar":      lambda m: (m["ax"] <= 1.2) & (m["ay"] <= m["r"]),
 "rising":     lambda m: (np.abs(m["dx"] + m["dy"]) <= 1.4) & (np.maximum(m["ax"], m["ay"]) <= m["r"]),
 "falling":    lambda m: (np.abs(m["dx"] - m["dy"]) <= 1.4) & (np.maximum(m["ax"], m["ay"]) <= m["r"]),
 "L":          lambda m: ((np.abs(m["dx"] + m["r"]) <= 1.2) & (m["ay"] <= m["r"])) | ((np.abs(m["dy"] - m["r"]) <= 1.2) & (m["ax"] <= m["r"])),
 "T":          lambda m: ((np.abs(m["dy"] + m["r"]) <= 1.2) & (m["ax"] <= m["r"])) | ((m["ax"] <= 1.2) & (m["ay"] <= m["r"])),
 "U":          lambda m: ((np.abs(m["ax"] - m["r"]) <= 1.2) & (m["ay"] <= m["r"])) | ((np.abs(m["dy"] - m["r"]) <= 1.2) & (m["ax"] <= m["r"])),
 "dot":        lambda m: m["d2"] <= (0.45 * m["r"])**2,
 "half disc":  lambda m: (m["d2"] <= m["r"]**2) & (m["dy"] <= 0),
 "checker":    lambda m: (np.maximum(m["ax"], m["ay"]) <= m["r"]) & ((m["dx"] > 0) ^ (m["dy"] > 0)),
 "corners":    lambda m: (np.maximum(m["ax"], m["ay"]) <= m["r"]) & (m["ax"] > 0.5 * m["r"]) & (m["ay"] > 0.5 * m["r"]),
})
CLASSES = list(SHAPES); K = len(CLASSES)
print(K, "classes:", ", ".join(CLASSES))

In [ ]:
def render(name, rng, jitter=True):
    """One 16x16 image in [0, 1]: shape = 1, background = 0, plus faint sensor noise."""
    cx = cy = 7.5; r = 5.0
    if jitter:
        cx, cy = 7.5 + rng.uniform(-2, 2), 7.5 + rng.uniform(-2, 2); r = rng.uniform(4.0, 6.0)
    img = SHAPES[name](_m(xx - cx, yy - cy, r)).astype(float)
    return np.clip(img + 0.03 * rng.standard_normal((16, 16)), 0, 1)

def make_dataset(n_per_class, seed):
    rng = np.random.default_rng(seed)
    X = np.stack([render(c, rng) for c in CLASSES for _ in range(n_per_class)]).astype(np.float32)
    y = np.repeat(np.arange(len(CLASSES)), n_per_class)
    perm = rng.permutation(len(y))
    return X[perm], y[perm]

In [ ]:
sr, n_rows, col_dur = 8000, 16, 0.06                      # week 10's sonifier: sample rate, image rows, seconds per column
t = np.arange(int(sr * col_dur)) / sr
freqs = np.logspace(np.log10(2800), np.log10(350), n_rows)  # row 0 (top) = highest pitch
W = np.sin(2 * np.pi * freqs[None, :] * t[:, None]).astype(np.float32)   # (480, 16)
NFFT = 512; fft_hz = np.fft.rfftfreq(NFFT, 1 / sr)
edges = np.logspace(np.log10(300), np.log10(3300), n_rows + 1)              # 16 log-spaced pitch bands
BANDS = [(fft_hz >= lo) & (fft_hz < hi) for lo, hi in zip(edges[:-1], edges[1:])]
window = np.hanning(len(t)).astype(np.float32)

def sonify(img):
    """(16, 16) image -> 1-D audio: columns become 60 ms snippets in time."""
    return (W @ img).T.reshape(-1)

def sound_map(audio):
    """audio -> (16, 16) spectrogram: energy per pitch band (rows, top = high) per snippet (cols)."""
    power = np.abs(np.fft.rfft(audio.reshape(16, -1) * window, n=NFFT)) ** 2
    e = np.sqrt(np.stack([power[:, b].sum(1) for b in BANDS], axis=1))
    return (e / (e.max() + 1e-8)).T[::-1]

def haptic_code(img, cells=4, levels=8):
    """(16, 16) -> (cells, cells) integers in [0, levels-1]: block average, then round (the hand-designed downsampler)."""
    pooled = img.reshape(cells, 16 // cells, cells, 16 // cells).mean(axis=(1, 3))
    return np.clip(np.round(pooled * (levels - 1)), 0, levels - 1).astype(np.int64)
print("sonifier ready:", W.shape, "| bands", len(BANDS))

In [ ]:
#@title 🔮 Predict A1: Load or regenerate the data. How many images will there be, and will the mean round-trip correlation (image vs its own spectrogram) be above 0.8? Will week10_plan.md be found?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A1", prediction, confidence)

In [ ]:
if os.path.exists(DATA_FILE):
    d = np.load(DATA_FILE)
    images, sound_vecs, haptic, labels = d["images"], d["sound"], d["haptic"].astype(np.int64), d["labels"]
    assert list(d["classes"]) == CLASSES, "class list differs from week 10's: regenerate"
    DATA_SOURCE = f"loaded {DATA_FILE} (seed {int(d['seed'])}, option {str(d['option'])})"
else:
    images, labels = make_dataset(cfg["n_per_class"], cfg["seed"])
    sound_vecs = np.stack([sound_map(sonify(im)).ravel() for im in images]).astype(np.float32)
    haptic = np.stack([haptic_code(im) for im in images])
    DATA_SOURCE = f"regenerated with week 10's code (seed {cfg['seed']}, {cfg['n_per_class']} per class)"
N = len(labels)
def corr(a, b): return float(np.corrcoef(a.ravel(), b.ravel())[0, 1])
round_trip = np.mean([corr(im, sv.reshape(16, 16)) for im, sv in zip(images[:200], sound_vecs[:200])])
print(DATA_SOURCE)
print(f"{N} images, {K} classes, sound vectors {sound_vecs.shape}, haptic codes {haptic.shape}, round-trip corr {round_trip:.3f}")
DEFAULT_PLAN = ["# Week 12 plan (default: week10_plan.md not found)", "lever        objective (contrastive image-sound loss)",
                "task/metric  20 rendered shapes; rank correlation between embedding closeness and confusion ordering",
                "baseline     pixel distance (trivial); pixel-loss encoder, same steps (strong)",
                f"prediction   A beats B by >= {X} rank correlation", f"ablation     {ablation_text}",
                f"stopping     {cfg['steps']} steps for every row, no early stopping", f"seeds        {len(cfg['seeds'])} per row",
                "decision     kill if gap < X or CI includes 0; pivot if ablation costs < X/2; scale up otherwise"]
if os.path.exists(PLAN_FILE): plan_text, PLAN_SOURCE = open(PLAN_FILE, encoding="utf-8").read(), "loaded " + PLAN_FILE
else: plan_text, PLAN_SOURCE = "\n".join(DEFAULT_PLAN), "default plan"
print("\n" + PLAN_SOURCE + ":\n" + plan_text)

In [ ]:
#@title ✍️ A1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A1", what_happened, surprise)

Surprise? On Colab the file is only there if you uploaded it: regeneration with the same seed gives the same images, so either path is fine, but the write-up must say which one happened (week 7's reproducibility row). The round trip is high but not 1: a 60 ms snippet blurs neighbouring pitches, which is exactly the loss a listener also suffers.

In [ ]:
#@title 🔮 Predict A2: A synthetic listener hears each sonified shape through 4 pitch bands x 8 time slices plus noise tuned so it scores about 50% (chance 5%) on 20-way identification. Which two shape pairs will it confuse most?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A2", prediction, confidence)

In [ ]:
def percept(sv):
    """(N, 256) sound vectors -> what a coarse listener keeps: 4 pitch bands x 8 time slices = 32 numbers."""
    m = sv.reshape(-1, 16, 16)                                    # (N, band, slice)
    return m.reshape(-1, 4, 4, 8, 2).mean(axis=(2, 4)).reshape(len(sv), -1)
P = percept(sound_vecs); P = (P - P.mean(0)) / (P.std(0) + 1e-6)
MEANS = np.stack([P[labels == k].mean(0) for k in range(K)])     # what each class sounds like on average
def listen(sigma, repeats, seed):
    """Noisy nearest-class-mean decisions, `repeats` passes over every image -> 20x20 confusion matrix."""
    r = np.random.default_rng(seed); C = np.zeros((K, K), int)
    for _ in range(repeats):
        noisy = P + sigma * r.standard_normal(P.shape)
        ans = ((noisy[:, None, :] - MEANS[None]) ** 2).sum(-1).argmin(1)
        np.add.at(C, (labels, ans), 1)
    return C
ladder = [0.5, 1, 1.5, 2, 3, 4, 6]; accs = [np.trace(listen(s, 3, 1)) / (3 * N) for s in ladder]
SIGMA = ladder[int(np.argmin([abs(a - 0.5) for a in accs]))]       # noise chosen so the listener scores about 50%
CONF = listen(SIGMA, cfg["listener_repeats"], 2); acc = np.trace(CONF) / CONF.sum()
PAIRS = [(a, b) for a in range(K) for b in range(a + 1, K)]; NAMES = [f"{CLASSES[a]}-{CLASSES[b]}" for a, b in PAIRS]
conf_pairs = np.array([CONF[a, b] + CONF[b, a] for a, b in PAIRS], float)
print(f"synthetic listener: sigma {SIGMA}, accuracy {acc * 100:.0f}% (chance 5%), {CONF.sum()} trials, {int(conf_pairs.sum())} confusions over {len(PAIRS)} pairs")
print("most confused pairs:", [NAMES[i] for i in np.argsort(-conf_pairs)[:5]])
plt.imshow(CONF, cmap="magma"); plt.xlabel("answered"); plt.ylabel("played"); plt.title("synthetic confusion matrix"); plt.show()

In [ ]:
#@title ✍️ A2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A2", what_happened, surprise)

Surprise? Whatever the pairs are, remember what this table is: a decision rule you wrote, applied to the sound. It gives the pipeline something with the right *shape* (190 counts) so that every later number can be computed end to end. Lesson 4's human study replaces `CONF` with real trials and nothing else in the notebook changes.

---
## Part B · Trivial baselines (20 min)

**Words used in this part.** *class-pair distance*: distance between the average representation of shape a and of shape b, one number per pair (190 pairs). *positions*: each value replaced by its place in its own list (1 = smallest; ties share the average place). *rank correlation*: the plain correlation of two lists of positions; +1 means the same ordering. *agreement*: rank correlation between model **closeness** (minus distance) and confusion counts, so a positive number means "close pairs are confused pairs". *bootstrap interval*: resample the 190 pairs with replacement, recompute the difference A − B each time, report the middle 95%.

In [ ]:
def class_means(X): return np.stack([X[labels == k].reshape((labels == k).sum(), -1).mean(0) for k in range(K)])
def pair_dists(M, cosine=False):
    if cosine: M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-8)
    return np.array([1 - M[a] @ M[b] if cosine else np.linalg.norm(M[a] - M[b]) for a, b in PAIRS])
def positions(v):
    """1 = smallest; tied values share the average position."""
    v = np.asarray(v, float); order = np.argsort(v, kind="stable"); r = np.empty(len(v)); r[order] = np.arange(1, len(v) + 1)
    _, inv = np.unique(v, return_inverse=True); return np.bincount(inv, r)[inv] / np.bincount(inv)[inv]
def rank_corr(u, v): return float(np.corrcoef(positions(u), positions(v))[0, 1])
def agreement(dist): return rank_corr(-dist, conf_pairs)          # closeness vs confusions: +1 = same ordering
def boot_ci(dists_a, dists_b, n=cfg["boots"], seed=3):
    """95% interval on mean-over-seeds(rho_A) - mean-over-seeds(rho_B), resampling the 190 pairs."""
    r = np.random.default_rng(seed); diffs = []
    for _ in range(n):
        idx = r.integers(0, len(PAIRS), len(PAIRS)); c = conf_pairs[idx]
        ra = np.mean([rank_corr(-d[idx], c) for d in dists_a]); rb = np.mean([rank_corr(-d[idx], c) for d in dists_b])
        diffs.append(ra - rb)
    return float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))
print("helpers ready:", len(PAIRS), "pairs | bootstrap resamples:", cfg["boots"])

In [ ]:
#@title 🔮 Predict B1: Agreement (rank correlation) between the confusion table and (a) pixel distance, (b) fixed-mapping sound-vector distance. Which is higher, and is either above 0.4?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
d_pixel = pair_dists(class_means(images)); d_sound = pair_dists(class_means(sound_vecs)); d_haptic = pair_dists(class_means(haptic.astype(np.float32)))
BASE = {"B trivial: pixel distance": agreement(d_pixel), "B fixed: sound-vector distance": agreement(d_sound), "haptic-code distance": agreement(d_haptic)}
for k, v in BASE.items(): print(f"{k:32s} rho = {v:+.2f}")
lo, hi = boot_ci([d_sound], [d_pixel])
print(f"sound - pixel: {BASE['B fixed: sound-vector distance'] - BASE['B trivial: pixel distance']:+.2f}  95% CI [{lo:+.2f}, {hi:+.2f}]")
fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
for a, d, t in zip(ax, [d_pixel, d_sound], ["pixel", "sound-vector"]):
    a.scatter(positions(-d), positions(conf_pairs), s=8); a.set_xlabel(f"{t} closeness position"); a.set_ylabel("confusion position")
plt.tight_layout(); plt.show()

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

Surprise? The fixed-mapping distance is the baseline an ML collaborator will propose first ("what baseline would make this go away?"): no learning at all, just the sonifier and a ruler. It goes into the write-up as a row whether or not it wins. Note that the listener was *built* from the sound map, so this rung is stronger here than it would be against people.

---
## Part C · Train once and read the curves (35 min)

**Words used in this part.** *encoder*: a small network from 256 numbers to a 32-d unit-length latent; one for images, one for spectrograms. *InfoNCE*: pick your partner out of the batch, both directions; chance = ln 64 = 4.16. *held-out retrieval*: on the 25% of pairs never trained on, the share of images whose own sound is the nearest held-out sound. *latent sd*: spread of the unit-length latents across the held-out set, averaged over the 32 dimensions; alive ≈ 0.18, collapsed < 0.02. *temperature*: τ = 0.07, fixed.

In [ ]:
class Enc(nn.Module):
    def __init__(self, d_in=256, d=32, hidden=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(), nn.Linear(hidden, d))
    def forward(self, x): return F.normalize(self.net(x.reshape(len(x), -1)), dim=1)

def info_nce(za, zb, tau=0.07):
    """Pick your partner out of the batch, both directions. Chance = ln(batch)."""
    s = za @ zb.T / tau; tgt = torch.arange(len(za))
    return (F.cross_entropy(s, tgt) + F.cross_entropy(s.T, tgt)) / 2

@torch.no_grad()
def retrieval(fa, fb, Xa, Xb):
    za, zb = fa(Xa), fb(Xb); return float(((za @ zb.T).argmax(1) == torch.arange(len(za))).float().mean()) * 100

@torch.no_grad()
def latent_sd(f, X): return float(f(X).std(0).mean())

def augment(x, g):
    """The ablation's second view: shift by up to 2 px and add noise. No sound involved."""
    dx, dy = [int(v) for v in torch.randint(-2, 3, (2,), generator=g)]
    return torch.roll(x, (dy, dx), dims=(1, 2)) + 0.05 * torch.randn(x.shape, generator=g)
print("encoder parameters:", sum(p.numel() for p in Enc().parameters()), "| chance loss ln 64 =", round(math.log(64), 2))

In [ ]:
def split(seed):
    g = torch.Generator().manual_seed(seed); perm = torch.randperm(N, generator=g); n_tr = int(0.75 * N)
    return g, perm[:n_tr], perm[n_tr:]

def train_contrastive(Xa, Xb, seed, steps=None, log_every=50, ablation=False):
    """Two encoders, InfoNCE, fixed steps. ablation=True ignores Xb: the second view is an augmented image."""
    steps = steps or cfg["steps"]; torch.manual_seed(seed); g, tr, te = split(seed); fa, fb = Enc(), Enc()
    opt = torch.optim.Adam(list(fa.parameters()) + list(fb.parameters()), lr=3e-3); hist = []
    xb_te = augment(Xa[te], torch.Generator().manual_seed(99)) if ablation else Xb[te]
    for step in range(1, steps + 1):
        idx = tr[torch.randperm(len(tr), generator=g)[:64]]
        xb = augment(Xa[idx], g) if ablation else Xb[idx]
        loss = info_nce(fa(Xa[idx]), fb(xb)); opt.zero_grad(); loss.backward(); opt.step()
        if step % log_every == 0 or step == steps:
            hist.append((step, loss.item(), retrieval(fa, fb, Xa[te], xb_te), latent_sd(fa, Xa[te])))
    return fa, fb, hist

IMG, SND = torch.tensor(images), torch.tensor(sound_vecs); N_TE = N - int(0.75 * N)
print(f"train / held-out: {N - N_TE} / {N_TE} | steps {cfg['steps']} | batch 64 | tau 0.07 | held-out chance {100 / N_TE:.1f}%")

In [ ]:
#@title 🔮 Predict C1: Train A (image<->sound) once, seed 0. Final training loss vs 4.16? Peak held-out retrieval (chance is 1 / held-out count)? Does retrieval fall again before the last step (memorising)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
t0 = time.time(); fa, fb, hist = train_contrastive(IMG, SND, seed=0)
for step, loss, ret, sd in hist[::max(1, len(hist) // 6)]:
    print(f"step {step:5d}  loss {loss:.2f}  held-out retrieval {ret:4.0f}%  latent sd {sd:.2f}")
steps_, losses, rets, sds = zip(*hist)
fig, ax = plt.subplots(figsize=(7, 3.2)); ax.plot(steps_, losses, label="train loss"); ax.axhline(math.log(64), ls=":", c="grey", label="chance ln 64")
ax.set_ylabel("InfoNCE"); ax.set_xlabel("step"); ax2 = ax.twinx(); ax2.plot(steps_, rets, "--", c="C1", label="held-out retrieval %"); ax2.set_ylabel("%")
ax.legend(loc="upper left"); ax2.legend(loc="upper right"); plt.show()
print(f"peak held-out retrieval {max(rets):.0f}% at step {steps_[int(np.argmax(rets))]}, final {rets[-1]:.0f}% | chance {100 / N_TE:.1f}% | latent sd {sds[-1]:.2f} | {time.time() - t0:.0f}s")

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

Reading it: the loss must sit clearly below 4.16 (else wiring: shuffled pairs, τ = 1, a huge learning rate); the held-out curve is the number you report, and if it peaked and fell you are memorising (fixed steps: report the final number, fix data or latent size for every row, never stop at the peak); latent sd near 0 with the loss at chance is collapse.

---
## Part D · Four rows × seeds (45 min)

**Words used in this part.** *row*: one method in the table; each row is run once per seed with the same steps and batch. *B strong*: an autoencoder with the same encoder, trained on pixel (reconstruction) loss; its 32-d bottleneck is the embedding. *ablation*: A minus the sound side: contrastive image ↔ augmented image, the sharpened ablation from A0. *gap*: mean over seeds of A's agreement minus B's. *bootstrap interval*: the 95% range of that gap when the 190 pairs are resampled.

In [ ]:
class AE(nn.Module):
    def __init__(self, d=32, hidden=128):
        super().__init__(); self.enc = Enc(256, d, hidden)
        self.dec = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 256), nn.Sigmoid())
    def forward(self, x): return self.dec(self.enc(x)).reshape(-1, 16, 16)

def train_ae(X, seed, steps=None):
    """B strong: same encoder, same steps and batch, pixel (reconstruction) loss instead of InfoNCE."""
    steps = steps or cfg["steps"]; torch.manual_seed(seed); g, tr, te = split(seed); ae = AE()
    opt = torch.optim.Adam(ae.parameters(), lr=3e-3)
    for step in range(steps):
        idx = tr[torch.randperm(len(tr), generator=g)[:64]]
        loss = F.mse_loss(ae(X[idx]), X[idx]); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): held = float(F.mse_loss(ae(X[te]), X[te]))
    return ae.enc, held

@torch.no_grad()
def latent_pair_dists(enc, X): return pair_dists(class_means(enc(X).numpy()), cosine=True)
ROWS = ["B strong: pixel-loss encoder", "A: contrastive image<->sound", "ablation: contrastive image<->image"]
def run_row(row, seed):
    if row.startswith("B strong"): enc, _ = train_ae(IMG, seed); return latent_pair_dists(enc, IMG)
    fa, fb, _ = train_contrastive(IMG, SND, seed, ablation=row.startswith("ablation")); return latent_pair_dists(fa, IMG)
print("chance MSE (predict the mean image):", round(float(((IMG - IMG.mean(0)) ** 2).mean()), 3), "| rows:", len(ROWS), "| seeds:", cfg["seeds"])

In [ ]:
#@title 🔮 Predict D1: Over the seeds: A's agreement minus B strong's, in rank correlation: below 0.1, 0.1-0.2, or >= 0.2 (= X)? And the ablation cost (A minus image<->image): near 0 or >= 0.1?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", prediction, confidence)

In [ ]:
t0 = time.time(); DISTS, RHO = {r: [] for r in ROWS}, {r: [] for r in ROWS}
for row in ROWS:
    for s in cfg["seeds"]:
        d = run_row(row, s); DISTS[row].append(d); RHO[row].append(agreement(d))
    print(f"{row:38s} rho {np.mean(RHO[row]):+.2f} ± {np.std(RHO[row], ddof=1):.2f}  seeds {[round(v, 2) for v in RHO[row]]}  ({time.time() - t0:.0f}s)")
for k in ["B trivial: pixel distance", "B fixed: sound-vector distance"]: print(f"{k:38s} rho {BASE[k]:+.2f} (no training, no seed)")

In [ ]:
A, B, ABL = ROWS[1], ROWS[0], ROWS[2]
GAP = float(np.mean(RHO[A]) - np.mean(RHO[B])); LO, HI = boot_ci(DISTS[A], DISTS[B])
ABL_COST = float(np.mean(RHO[A]) - np.mean(RHO[ABL])); ABL_LO, ABL_HI = boot_ci(DISTS[A], DISTS[ABL])
SD_POOL = float(np.sqrt(np.mean([np.var(v, ddof=1) for v in RHO.values()])))
print(f"A - B strong = {GAP:+.3f}  95% CI [{LO:+.2f}, {HI:+.2f}]  (pre-registered X = {X})")
print(f"ablation cost (A - image<->image) = {ABL_COST:+.3f}  95% CI [{ABL_LO:+.2f}, {ABL_HI:+.2f}]")
print(f"seed spread (pooled sd) {SD_POOL:.3f} -> effect size of the gap {GAP / (SD_POOL + 1e-9):+.1f}")
names = ["B trivial\npixels", "B fixed\nsound vec", "B strong\npixel-loss enc", "A\ncontrastive", "ablation\nimg<->img"]
vals = [[BASE["B trivial: pixel distance"]], [BASE["B fixed: sound-vector distance"]], RHO[B], RHO[A], RHO[ABL]]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for i, v in enumerate(vals):
    ax.bar(i, np.mean(v), color="lightgrey", width=0.6); ax.scatter([i] * len(v), v, c="C0", zorder=3)
ax.axhline(np.mean(RHO[B]) + X, ls="--", c="C3", label=f"B strong + X ({X})"); ax.set_xticks(range(5)); ax.set_xticklabels(names, fontsize=8)
ax.set_ylabel("rank correlation with confusions"); ax.legend(); plt.tight_layout(); plt.savefig("week12_plot.png", dpi=120); plt.show()
print("saved week12_plot.png: rows on x, agreement on y, one dot per seed, dashed line = what A had to reach")

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

In [ ]:
#@title ✍️ A0: did A clear the pre-registered X against B strong? Was that a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A0", what_happened or f"A - B = {GAP:+.2f}, CI [{LO:+.2f}, {HI:+.2f}], X = {X}", surprise)

Surprise? Three things can be true at once: A can beat B strong, lose to B fixed (the sonifier plus a ruler), and the ablation can cost nothing. Each is a different sentence in the write-up, and only the first one is the claim. In SMOKE mode (240 images, 3 seeds, 300 steps) the interval will be wide; the decision the notebook prints is a rehearsal of the rule, and the full run is the one that counts.

---
## Part E · Optional: the codec alternative (25 min)

**Words used in this part.** *bottleneck*: the 16 numbers in the middle of the autoencoder, one per tactor. *quantization*: each of the 16 rounded to one of 8 levels, 48 bits per frame, the same bandwidth as week 10's hand-designed downsampler. *straight-through*: `z + (q - z).detach()`, rounded in the forward pass, identity in the backward pass. *probe*: a small classifier trained on the codes; its held-out accuracy says how much shape identity 48 bits kept. Set `RUN_CODEC = False` in the config cell to skip this part.

In [ ]:
#@title 🔮 Predict E1: A learned 4x4 x 8-level code (straight-through) vs the hand-designed downsampler at the same 48 bits: which gives the higher probe accuracy, and by how many points (week 10's codec prediction said >= 5)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
if RUN_CODEC:
    def quantize(z, levels=8):
        q = torch.round(z * (levels - 1)) / (levels - 1)
        return z + (q - z).detach()                     # forward: rounded; backward: straight through
    class Codec(nn.Module):
        def __init__(self, cells=4, levels=8, hidden=128):
            super().__init__(); self.levels = levels
            self.enc = nn.Sequential(nn.Linear(256, hidden), nn.ReLU(), nn.Linear(hidden, cells * cells), nn.Sigmoid())
            self.dec = nn.Sequential(nn.Linear(cells * cells, hidden), nn.ReLU(), nn.Linear(hidden, 256), nn.Sigmoid())
        def code(self, x): return quantize(self.enc(x.reshape(len(x), -1)), self.levels)
        def forward(self, x): return self.dec(self.code(x)).reshape(-1, 16, 16)
    torch.manual_seed(0); g, tr, te = split(0); codec = Codec(); opt = torch.optim.Adam(codec.parameters(), lr=3e-3)
    for step in range(cfg["steps"]):
        idx = tr[torch.randperm(len(tr), generator=g)[:64]]
        loss = F.mse_loss(codec(IMG[idx]), IMG[idx]); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): learned_codes = (codec.code(IMG) * 7).round().numpy(); held_mse = float(F.mse_loss(codec(IMG[te]), IMG[te]))
    hand_mse = float((((haptic / 7.0).repeat(4, axis=1).repeat(4, axis=2) - images) ** 2).mean())
    print(f"48-bit codes | held-out MSE learned {held_mse:.4f} vs hand downsampler {hand_mse:.4f} | grad_fn of the code: {codec.code(IMG[:2]).grad_fn}")
else: print("codec part skipped (RUN_CODEC = False)")

In [ ]:
def probe_accuracy(Xf, y, seed, steps=200, hidden=64):
    """A small classifier on one representation, 75/25 split by seed; held-out accuracy in %."""
    torch.manual_seed(seed); g, tr, te = split(seed); Xt, yt = torch.tensor(Xf, dtype=torch.float32), torch.tensor(y)
    Xt = (Xt - Xt.mean(0)) / (Xt.std(0) + 1e-6)
    net = nn.Sequential(nn.Linear(Xt.shape[1], hidden), nn.ReLU(), nn.Linear(hidden, K)); opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for _ in range(steps):
        idx = tr[torch.randperm(len(tr), generator=g)[:64]]; loss = F.cross_entropy(net(Xt[idx]), yt[idx]); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): return float((net(Xt[te]).argmax(1) == yt[te]).float().mean()) * 100
if RUN_CODEC:
    reps = {"hand 4x4x8": haptic.reshape(N, -1).astype(np.float32), "learned 4x4x8": learned_codes.reshape(N, -1)}
    res = {k: [probe_accuracy(v, labels, s) for s in cfg["seeds"]] for k, v in reps.items()}
    for k, v in res.items(): print(f"{k:14s} probe accuracy {np.mean(v):5.1f} ± {np.std(v, ddof=1):.1f}  (chance 5%)")
    CODEC_GAP = float(np.mean(res["learned 4x4x8"]) - np.mean(res["hand 4x4x8"])); print(f"learned - hand = {CODEC_GAP:+.1f} points (week 10's codec prediction: >= 5)")
else: CODEC_GAP = None; print("codec part skipped")

In [ ]:
#@title ✍️ E1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E1", what_happened, surprise)

Surprise? If the learned code wins, the pivot target exists and has a number. If it loses at 48 bits, the hand-designed downsampler is a strong baseline, which is worth one line in the write-up's ASK.

---
## Part F · The one-page write-up (30 min)

**Words used in this part.** *decision rule*: kill if the gap misses X or the interval includes 0; pivot if the gap clears X but the ablation cost less than X / 2; scale up otherwise. Written before the run, applied without discussion. *surprise*: the ledger row with the largest gap between prediction and result. The cell fills the template from the numbers above, saves `week12_writeup.md`, and prints it: download it with `week12_plot.png`.

In [ ]:
rule = "kill" if (GAP < X or LO <= 0) else ("pivot" if ABL_COST < X / 2 else "scale up")
why = {"kill": f"gap {GAP:+.2f} < X {X}" if GAP < X else f"CI [{LO:+.2f}, {HI:+.2f}] includes 0",
       "pivot": f"gap cleared X but ablation cost {ABL_COST:+.2f} < X/2 = {X / 2}", "scale up": f"gap >= X, CI above 0, ablation cost {ABL_COST:+.2f} >= X/2"}[rule]
done = [(t, r) for t, r in _LEDGER.items() if r["actual"]]
big = [x for x in done if x[1]["surprise"] == "big"] or sorted(done, key=lambda x: -x[1]["confidence"])[:1]
surprise = "; ".join(f"{t}: predicted '{r['prediction']}' at {r['confidence']}%, got '{r['actual']}'" for t, r in big) or "(no recorded surprises)"
rows = "\n".join(f"          {k:38s} {np.mean(v):+.2f} ± {np.std(v, ddof=1):.2f} (n={len(v)})" for k, v in RHO.items())
page = f"""# Week 12 result  (option: contrastive, SMOKE={SMOKE}, {time.strftime('%Y-%m-%d')})
CLAIM     A (contrastive image<->sound) beats B (pixel-loss encoder) by >= {X} rank correlation with the confusion table
SETUP     {DATA_SOURCE}; {PLAN_SOURCE}; N={N}; {len(ROWS)} trained rows x {len(cfg['seeds'])} seeds x {cfg['steps']} steps; {(time.time() - T_START) / 60:.1f} min CPU
          synthetic listener (4 bands x 8 slices, sigma {SIGMA}, {CONF.sum()} trials) stands in for the human study
RESULT
{rows}
          B trivial: pixel distance {BASE['B trivial: pixel distance']:+.2f}; B fixed: sound-vector distance {BASE['B fixed: sound-vector distance']:+.2f}
          A - B strong = {GAP:+.2f}  95% CI [{LO:+.2f}, {HI:+.2f}]; ablation cost {ABL_COST:+.2f}; seed spread {SD_POOL:.2f}
PLOT      week12_plot.png (rows on x, agreement on y, seeds as dots, dashed line = B strong + X)
SURPRISE  {surprise}
DECISION  {rule.upper()}  (rule: {why})
ASK       what baseline would make this go away? not yet a row: a random encoder of the same size; a classifier's penultimate layer"""
open("week12_writeup.md", "w", encoding="utf-8").write(page + "\n"); print(page)

---
## After the session

1. Download `week12_writeup.md` and `week12_plot.png`; paste the prediction table below into your insight ledger, marking the biggest surprise.
2. `python track.py done week-12/build --rating <1-5> --minutes 180 --note "biggest surprise"`
3. Canvas step 10: send the page to an ML collaborator with the ASK line. Log their baseline next to `build-step10`, and whether it is already one of your rows.

In [ ]:
reveal()